# অগ্রাধিকার সদস্য মিডলওয়্যার সহ হোটেল বুকিং

এই নোটবুকটি মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক ব্যবহার করে **ফাংশন-ভিত্তিক মিডলওয়্যার** প্রদর্শন করে। আমরা শর্তাধীন কর্মপ্রবাহ উদাহরণের উপরে একটি মিডলওয়্যার স্তর যোগ করি যা অগ্রাধিকার সদস্যদের বিশেষ সুবিধা দেয়।

## আপনি যা শিখবেন:
1. **ফাংশন-ভিত্তিক মিডলওয়্যার**: ফাংশনের ফলাফল অবরোধ ও পরিবর্তন করা
2. **কনটেক্সট অ্যাক্সেস**: কার্যকরী হওয়ার পর `context.result` পড়া ও পরিবর্তন করা
3. **ব্যবসায়িক লজিক বাস্তবায়ন**: অগ্রাধিকার সদস্য সুবিধাসমূহ
4. **ফলাফল ওভাররাইড**: ব্যবহারকারীর অবস্থা অনুযায়ী ফাংশনের ফলাফল পরিবর্তন করা
5. **একই কর্মপ্রবাহ, ভিন্ন ফলাফল**: মিডলওয়্যার নির্দেশিত আচরণ পরিবর্তন

## মিডলওয়্যার সহ কর্মপ্রবাহ স্থাপত্য:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## শর্তাধীন কর্মপ্রবাহ থেকে মূল পার্থক্য:

**মিডলওয়্যার ছাড়া** (14-conditional-workflow.ipynb):
- প্যারিসে কোন ঘর নেই → alternative_agent এ রাউট করুন

**মিডলওয়্যার সহ** (এই নোটবুক):
- নিয়মিত ব্যবহারকারী + প্যারিস → কোন ঘর নেই → alternative_agent এ রাউট করুন
- অগ্রাধিকার ব্যবহারকারী + প্যারিস → 🌟 মিডলওয়্যার ওভাররাইড! → ঘর পাওয়া গেছে → booking_agent এ রাউট করুন

## প্রয়োজনীয়তা:
- মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক ইনস্টল করা হয়েছে
- শর্তাধীন কর্মপ্রবাহের ধারণা রয়েছে (দেখুন 14-conditional-workflow.ipynb)
- GitHub টোকেন বা OpenAI API কী
- মিডলওয়্যার প্যাটার্নের মৌলিক ধারণা


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Step 1: সংরचित আউটপুটগুলির জন্য Pydantic মডেল সংজ্ঞায়িত করুন

এই মডেলগুলি সেই **স্কিমা** সংজ্ঞায়িত করে যা এজেন্টরা ফেরত দেবে। আমরা একটি `priority_override` ক্ষেত্র যোগ করেছি যা মিডলওয়্যার কখন উপলব্ধতার ফলাফল পরিবর্তন করে তা ট্র্যাক করে।


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ধাপ ২: অগ্রাধিকার সদস্যদের ডাটাবেস সংজ্ঞায়িত করুন

এই ডেমোর জন্য, আমরা একটি অগ্রাধিকার সদস্যপদ ডাটাবেস সিমুলেট করব। উৎপাদনে, এটি একটি প্রকৃত ডাটাবেস অথবা API থেকে তথ্য অনুসন্ধান করবে।

**অগ্রাধিকার সদস্যরা:**
- `alice@example.com` - ভিআইপি সদস্য
- `bob@example.com` - প্রিমিয়াম সদস্য  
- `priority_user` - পরীক্ষামূলক অ্যাকাউন্ট


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Step 3: হোটেল বুকিং টুল তৈরি করুন

শর্তাধীন ওয়ার্কফ্লোর মতো, তবে এখন এটি মিডলওয়্যার দ্বারা আটকানো হবে!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ধাপ ৪: 🌟 অগ্রাধিকার চেক মिडলওয়্যার তৈরি করুন (মূল বৈশিষ্ট্য!)

এটি এই নোটবুকের **মূল কার্যকারিতা**। মিডলওয়্যারটি:

১. **হস্তক্ষেপ করে** hotel_booking ফাংশন কল-এ
২. `next(context)` কল করে ফাংশনটি স্বাভাবিকভাবে **চালায়**
৩. `context.result`-এ ফলাফল **পরীক্ষা করে**
৪. যদি ব্যবহারকারী অগ্রাধিকারপূর্ণ হয় এবং কোনো রুম না থাকে তবে ফলাফল **অপরিবর্তিত করে**
৫. পরিবর্তিত ফলাফলটি আবার এজেন্টের কাছে **ফিরিয়ে দেয়**

**মূল প্যাটার্ন:**
```python
async def my_middleware(context, next):
    await next(context)  # ফাংশন সম্পাদন করুন
    # এখন context.result ফাংশনের আউটপুট ধারণ করছে
    if some_condition:
        context.result = new_value  # ওভাররাইড!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## পদক্ষেপ ৫: রাউটিংয়ের জন্য কন্ডিশন ফাংশন সংজ্ঞায়িত করুন

সর্বনিম্ন অবস্থার ফাংশনগুলি কন্ডিশনাল ওয়ার্কফ্লোর মতোই - তারা স্ট্রাকচার্ড আউটপুট পরীক্ষা করে রাউটিং নির্ধারণ করে।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Step 6: কাস্টম ডিসপ্লে এক্সিকিউটর তৈরি করুন

আগের একই এক্সিকিউটর - চূড়ান্ত ওয়ার্কফ্লো আউটপুট প্রদর্শন করে।


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Step 7: পরিবেশের পরিবর্তনশীল লোড করুন

LLM ক্লায়েন্ট (GitHub মডেল বা OpenAI) কনফিগার করুন।


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Step 8: মিডলওয়্যার সহ AI এজেন্ট তৈরি করা

**মূল পার্থক্য:** availability_agent তৈরি করার সময়, আমরা `middleware` প্যারামিটার পাঠাই!

এখন এভাবে আমরা priority_check_middleware এজেন্টের ফাংশন ইনভোশনের পাইপলাইনে ইনজেক্ট করি।


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: ওয়ার্কফ্লো তৈরি করুন

আগের মতো একই ওয়ার্কফ্লো কাঠামো - প্রাপ্যতার উপর ভিত্তি করে শর্তাধীন রাউটিং।


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: টেস্ট কেস ১ - প্যারিসে সাধারণ ব্যবহারকারী (ওভাররাইড নেই)

একজন সাধারণ ব্যবহারকারী প্যারিস বুক করার চেষ্টা করে → কোনও রুম নেই → alternative_agent-এ রুট করা হয়


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 প্যারিসে অগ্রাধিকার ব্যবহারকারী (ওভাররাইড সহ!)

একজন অগ্রাধিকার সদস্য প্যারিস বুক করার চেষ্টা করেন → শুরুতে কোনও রুম নেই → 🌟 মিডলওয়্যার ওভাররাইড করে! → বুকিং_এজেন্টের কাছে পাঠানো হয়

**এটাই মিডলওয়্যারের ক্ষমতার মূল প্রদর্শনী!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## পদক্ষেপ ১২: টেস্ট কেস ৩ - স্টকহোমে প্রাধান্য ব্যবহারকারী (ইতোমধ্যে উপলব্ধ)

প্রাধান্য ব্যবহারকারী স্টকহোম চেষ্টা করে → কক্ষ উপলব্ধ → ওভাররাইডের প্রয়োজন নেই → booking_agent এ রুট করে

এটি দেখায় যে, middleware শুধুমাত্র তখনই কাজ করে যখন প্রয়োজন হয়!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## মূল ধারণা এবং মিডলওয়্যার ধারনা

### ✅ আপনি যা শিখেছেন:

#### **১. ফাংশন-ভিত্তিক মিডলওয়্যার প্যাটার্ন**

মিডলওয়্যার একটি সাধারণ async ফাংশন ব্যবহার করে ফাংশন কলগুলোকে ইন্টারসেপ্ট করে:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ফাংশন কার্যকরোর আগে
    print("Intercepting...")
    
    # ফাংশন কার্যকর করো
    await next(context)
    
    # ফাংশন কার্যকরোর পরে - ফলাফল পরিদর্শন করো
    if context.result:
        # প্রয়োজনে ফলাফল পরিবর্তন করো
        context.result = modified_value
```

#### **২. প্রেক্ষাপট অ্যাক্সেস এবং ফলাফল ওভাররাইড**

- `context.function` - কল করা ফাংশনে অ্যাক্সেস
- `context.arguments` - ফাংশনের আর্গুমেন্ট পড়ুন
- `context.kwargs` - অতিরিক্ত প্যারামিটার অ্যাক্সেস করুন
- `await next(context)` - ফাংশন চালান
- `context.result` - ফাংশনের আউটপুট পড়ুন/পরিমার্জন করুন

#### **৩. ব্যবসায়িক লজিক বাস্তবায়ন**

আমাদের মিডলওয়্যার প্রাধান্য সদস্য সুবিধাগুলো বাস্তবায়ন করে:
- **নিয়মিত ব্যবহারকারীরা**: কোন পরিবর্তন নয়, স্ট্যান্ডার্ড ওয়ার্কফ্লো
- **প্রাধান্য ব্যবহকারীরা**: "কোন উপলব্ধতা নেই" ওভাররাইড করে "উপলব্ধ" দেখায়
- **শর্তাধীন লজিক**: শুধুমাত্র প্রয়োজন হলে ওভাররাইড করে

#### **৪. একই ওয়ার্কফ্লো, আলাদা ফলাফল**

মিডলওয়্যারের শক্তি:
- ✅ ওয়ার্কফ্লো কাঠামোতে কোন পরিবর্তন নয়
- ✅ টুল ফাংশনে কোন পরিবর্তন নয়
- ✅ শর্তাধীন রাউটিং লজিকে কোন পরিবর্তন নয়
- ✅ শুধুমাত্র মিডলওয়্যার → আলাদা আচরণ!

### 🚀 বাস্তব জীবনের প্রয়োগসমূহ:

১. **ভিআইপি/প্রিমিয়াম ফিচার**
   - প্রিমিয়াম ব্যবহারকারীদের জন্য রেট লিমিট ওভাররাইড করা
   - রিসোর্সে প্রাধান্য ভিত্তিক অ্যাক্সেস দেয়া
   - ডাইনামিকভাবে প্রিমিয়াম ফিচার আনলক করা

২. **এ/বি টেস্টিং**
   - ব্যবহারকারীদের ভিন্ন ভিন্ন ইমপ্লেমেন্টেশনে রাউট করা
   - নির্দিষ্ট ব্যবহারকারীদের সাথে নতুন ফিচার পরীক্ষা করা
   - ধাপে ধাপে ফিচার রোলআউট

৩. **সিকিউরিটি এবং কমপ্লায়েন্স**
   - ফাংশন কল যাচাই করা
   - সংবেদনশীল অপারেশনগুলো ব্লক করা
   - ব্যবসায়িক নিয়ম প্রয়োগ করা

৪. **পারফরম্যান্স অপ্টিমাইজেশন**
   - নির্দিষ্ট ব্যবহারকারীর জন্য ক্যাশে ফলাফল রাখা
   - সম্ভব হলে ব্যয়বহুল অপারেশন বাদ দেওয়া
   - ডাইনামিক রিসোর্স বরাদ্দ

৫. **এরর হ্যান্ডলিং এবং রিট্রাই**
   - এররগুলো গ্রেসফুলি ধরুন এবং হ্যান্ডল করুন
   - রিট্রাই লজিক বাস্তবায়ন করুন
   - বিকল্প ইমপ্লিমেন্টেশনে ফালব্যাক দিন

৬. **লগিং এবং মনিটরিং**
   - ফাংশন এক্সিকিউশন সময় ট্র্যাক করুন
   - প্যারামিটার এবং ফলাফল লগ করুন
   - ব্যবহার প্যাটার্ন মনিটর করুন

### 🔑 ডেকোরেটর থেকে মূল পার্থক্যসমূহ:

| বৈশিষ্ট্য | ডেকোরেটর | মিডলওয়্যার |
|---------|-----------|------------|
| **স্কোপ** | একক ফাংশন | এজেন্টের সব ফাংশন |
| **নমনীয়তা** | সংজ্ঞায় নির্দিষ্ট | রানটাইমে ডাইনামিক |
| **প্রেক্ষাপট** | সীমিত | পূর্ণ এজেন্ট প্রেক্ষাপট |
| **কম্পোজিশন** | একাধিক ডেকোরেটর | মিডলওয়্যার পাইপলাইন |
| **এজেন্ট সচেতন** | না | হ্যাঁ (এজেন্ট স্টেটে অ্যাক্সেস) |

### 📚 কখন মিডলওয়্যার ব্যবহার করবেন:

✅ **মিডলওয়্যার ব্যবহার করুন যখন:**
- ব্যবহারকারী/সেশন অবস্থার উপর ভিত্তি করে আচরণ পরিবর্তন করতে চান
- একাধিক ফাংশনে লজিক প্রয়োগ করতে চান
- এজেন্ট-স্তরের প্রেক্ষাপটে অ্যাক্সেস প্রয়োজন
- ক্রস-কাটিং বিষয়াদি (লগিং, অথ, ইত্যাদি) বাস্তবায়ন করছেন

❌ **মিডলওয়্যার ব্যবহার করবেন না যখন:**
- সহজ ইনপুট যাচাই (Pydantic ব্যবহার করুন)
- ফাংশন-নির্দিষ্ট লজিক (ফাংশনের মধ্যে রাখুন)
- এককালীন পরিবর্তন (সরাসরি ফাংশন পরিবর্তন করুন)

### 🎓 উন্নত প্যাটার্নসমূহ:

```python
# একাধিক মিডলওয়্যার (কার্যকরী ক্রম গুরুত্বপূর্ণ!)
middleware=[
    logging_middleware,      # প্রথমে লগগুলি
    auth_middleware,         # তারপর অথেনটিকেশন পরীক্ষা করে
    cache_middleware,        # তারপর ক্যাশ পরীক্ষা করে
    rate_limit_middleware,   # তারপর রেট সীমা নির্ধারণ করে
    priority_check_middleware  # অবশেষে অগ্রাধিকার পরীক্ষা
]

# শর্তসাপেক্ষ মিডলওয়্যার কার্যকরী
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ফলাফল সংশোধন করুন
    else:
        # সম্পূর্ণরূপে কার্যকরী এড়িয়ে যান
        context.result = cached_value
```

### 🔗 সম্পর্কিত ধারণাসমূহ:

- **এজেন্ট মিডলওয়্যার**: agent.run() কল ইন্টারসেপ্ট করে
- **ফাংশন মিডলওয়্যার**: টুল ফাংশন কল ইন্টারসেপ্ট করে (যা আমরা ব্যবহার করেছি!)
- **মিডলওয়্যার পাইপলাইন**: মিডলওয়্যার চেইন যা ক্রমান্বয়ে এক্সিকিউট হয়
- **প্রেক্ষাপট প্রোপাগেশন**: মিডলওয়্যার চেইনে স্টেট পাস করা


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
